In [1]:
import os
import pandas as pd
import boto3

try:
    import kagglehub
except:
    ! pip install kagglehub
    import kagglehub

try:
    import pyarrow
except:
    ! pip install pyarrow
    import pyarrow

## Helper Classes

In [2]:
class DataCollector:
    def __init__(self, str_bucket):
        self.str_bucket = str_bucket
        self.s3_client = boto3.client('s3')
        self.df_data = None

    def download_from_kaggle(self):
        print('Downloading Groceries dataset from Kaggle...')
        str_path = kagglehub.dataset_download('heeraldedhia/groceries-dataset')
        print(f'Dataset downloaded to: {str_path}')
        list_files = [f for f in os.listdir(str_path) if f.endswith('.csv')]
        if not list_files:
            raise FileNotFoundError('No CSV file found in downloaded dataset')
        str_csv_path = os.path.join(str_path, list_files[0])
        print(f'Loading CSV: {str_csv_path}')
        self.df_data = pd.read_csv(str_csv_path)
        return self.df_data

    def clean_data(self):
        if self.df_data is None:
            raise ValueError('Data not loaded. Call download_from_kaggle() first.')
        print('\nCleaning data...')
        # standardize column names
        self.df_data.columns = [c.strip().lower().replace(' ', '_') for c in self.df_data.columns]
        # rename for consistency
        dict_rename = {
            'member_number': 'member_id',
            'date': 'transaction_date',
            'itemdescription': 'item'
        }
        self.df_data = self.df_data.rename(columns=dict_rename)
        # parse date
        self.df_data['transaction_date'] = pd.to_datetime(self.df_data['transaction_date'], dayfirst=True)
        # strip whitespace from item names and lowercase
        self.df_data['item'] = self.df_data['item'].str.strip().str.lower()
        # drop nulls
        int_nulls = self.df_data.isnull().sum().sum()
        print(f'Total null values: {int_nulls}')
        self.df_data = self.df_data.dropna()
        # ensure member_id is int
        self.df_data['member_id'] = self.df_data['member_id'].astype('int64')
        # sort by date
        self.df_data = self.df_data.sort_values('transaction_date').reset_index(drop=True)
        print(f'Data shape after cleaning: {self.df_data.shape}')
        return self.df_data

    def get_summary_stats(self):
        if self.df_data is None:
            raise ValueError('Data not loaded.')
        print('\n=== SUMMARY STATISTICS ===')
        print(f'Total rows (item-level): {len(self.df_data):,}')
        print(f'Unique members: {self.df_data["member_id"].nunique():,}')
        print(f'Unique items: {self.df_data["item"].nunique():,}')
        print(f'Date range: {self.df_data["transaction_date"].min()} to {self.df_data["transaction_date"].max()}')
        # transactions = unique (member, date) pairs
        int_transactions = self.df_data.groupby(['member_id', 'transaction_date']).ngroups
        print(f'Total transactions (member + date): {int_transactions:,}')
        flt_avg_basket = len(self.df_data) / int_transactions
        print(f'Average items per transaction: {flt_avg_basket:.2f}')
        print(f'\nTop 10 items:')
        print(self.df_data['item'].value_counts().head(10))
        print(f'\nColumns: {list(self.df_data.columns)}')

    def save_to_s3(self, str_filename):
        if self.df_data is None:
            raise ValueError('Data not loaded.')
        str_s3_key = f'00_data_collection/{str_filename}'
        print(f'\nSaving to S3: s3://{self.str_bucket}/{str_s3_key}')
        from io import BytesIO
        buffer = BytesIO()
        self.df_data.to_parquet(buffer, index=False)
        buffer.seek(0)
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print(f'Successfully uploaded {str_filename} to S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')

## Constants

In [3]:
str_bucket = 'market-basket-analysis-demo'
str_task = '00_data_collection'

## Download and Prepare Data

In [4]:
cls_collector = DataCollector(str_bucket)
df_raw = cls_collector.download_from_kaggle()

100%|██████████| 257k/257k [00:00<00:00, 8.86MB/s]

Extracting files...
Dataset downloaded to: /home/ec2-user/.cache/kagglehub/datasets/heeraldedhia/groceries-dataset/versions/1
Loading CSV: /home/ec2-user/.cache/kagglehub/datasets/heeraldedhia/groceries-dataset/versions/1/Groceries_dataset.csv


In [5]:
print('First few rows of raw data:')
print(df_raw.head(10))
print(f'\nData shape: {df_raw.shape}')
print(f'\nColumn names and types:')
print(df_raw.dtypes)

First few rows of raw data:
   Member_number        Date   itemDescription
0           1808  21-07-2015    tropical fruit
1           2552  05-01-2015        whole milk
2           2300  19-09-2015         pip fruit
3           1187  12-12-2015  other vegetables
4           3037  01-02-2015        whole milk
5           4941  14-02-2015        rolls/buns
6           4501  08-05-2015  other vegetables
7           3803  23-12-2015        pot plants
8           2762  20-03-2015        whole milk
9           4119  12-02-2015    tropical fruit

Data shape: (38765, 3)

Column names and types:
Member_number       int64
Date               object
itemDescription    object
dtype: object


In [6]:
df_cleaned = cls_collector.clean_data()


Cleaning data...
Total null values: 0
Data shape after cleaning: (38765, 3)


In [7]:
cls_collector.get_summary_stats()


=== SUMMARY STATISTICS ===
Total rows (item-level): 38,765
Unique members: 3,898
Unique items: 167
Date range: 2014-01-01 00:00:00 to 2015-12-30 00:00:00
Total transactions (member + date): 14,963
Average items per transaction: 2.59

Top 10 items:
item
whole milk          2502
other vegetables    1898
rolls/buns          1716
soda                1514
yogurt              1334
root vegetables     1071
tropical fruit      1032
bottled water        933
sausage              924
citrus fruit         812
Name: count, dtype: int64

Columns: ['member_id', 'transaction_date', 'item']


In [8]:
cls_collector.save_to_s3('groceries.parquet')


Saving to S3: s3://market-basket-analysis-demo/00_data_collection/groceries.parquet
Successfully uploaded groceries.parquet to S3


## Completion

In [9]:
print('\n=== DATA COLLECTION COMPLETE ===')
print(f'File ready for next stage: s3://{str_bucket}/00_data_collection/groceries.parquet')


=== DATA COLLECTION COMPLETE ===
File ready for next stage: s3://market-basket-analysis-demo/00_data_collection/groceries.parquet
